# 🗄️ Course 10 — Data-Driven Decision Making in SQL

> **Platform:** DataCamp | **Track:** Associate Data Analyst in SQL  
> **Tool:** PostgreSQL | **Database:** MovieNow (online movie rental platform)

---

## 📋 About This Course

This course applies SQL to real business intelligence scenarios using a movie rental database. It covers KPI reporting, customer behavior analysis, advanced subqueries with EXISTS/IN, and OLAP operators (CUBE, ROLLUP, GROUPING SETS) for multi-dimensional analysis.

**Tables:** `renting`, `movies`, `customers`, `actors`, `actsin`

---

## 📚 Table of Contents

| Chapter | Topic |
|---------|-------|
| Chapter 1 | Introduction to Business Intelligence |
| Chapter 2 | Decision Making with Simple SQL Queries |
| Chapter 3 | Advanced SQL — Subqueries & EXISTS |
| Chapter 4 | OLAP Queries — CUBE, ROLLUP, GROUPING SETS |

---

## 📌 Chapter 1 — Introduction to Business Intelligence

---

### Exploring the `renting` Table
Understand the structure of movie rentals data.

In [ ]:
SELECT * FROM renting;           -- All columns
SELECT movie_id, rating FROM renting;  -- Columns needed for avg rating per movie
-- Note: rating column contains NULL values

### Working with Dates
Filter rentals by specific dates and date ranges.

In [ ]:
-- Rentals on October 9, 2018
SELECT * FROM renting WHERE date_renting = '2018-10-9';

-- Rentals April–August 2018, most recent first
SELECT * FROM renting
WHERE date_renting BETWEEN '2018-04-01' AND '2018-08-31'
ORDER BY date_renting DESC;

### Selecting Movies
Filter and sort the movies table.

In [ ]:
-- Movies that are NOT dramas
SELECT * FROM movies WHERE genre <> 'Drama';

-- Specific titles
SELECT * FROM movies WHERE title IN ('Showtime', 'Love Actually', 'The Fighter');

-- Order by renting price
SELECT * FROM movies ORDER BY renting_price ASC;

### Filter Rentals with Ratings
Select 2018 rentals that have a rating.

In [ ]:
SELECT * FROM renting
WHERE date_renting BETWEEN '2018-01-01' AND '2018-12-31'
  AND rating IS NOT NULL;

### Summarizing Customer Information
Count customers by birth decade, country, and distinct countries.

In [ ]:
-- Customers born in the 80s
SELECT COUNT(*) FROM customers
WHERE date_of_birth BETWEEN '1980-01-01' AND '1989-12-31';

-- Customers from Germany
SELECT COUNT(*) FROM customers WHERE country = 'germany';

-- Number of distinct countries
SELECT count(DISTINCT country) FROM customers;

### Rating Statistics for Movie 25
Report min, max, avg, and count of ratings for a specific movie.

In [ ]:
SELECT MIN(rating)   AS min_rating,
	   MAX(rating)   AS max_rating,
	   AVG(rating)   AS avg_rating,
	   COUNT(rating) AS number_ratings
FROM renting
WHERE movie_id = 25;

### Examining Annual Rentals (KPI Report)
Count rentals, average rating, and number of ratings since January 2019.

In [ ]:
SELECT 
	COUNT(*) AS number_renting,
	AVG(rating) AS average_rating, 
    COUNT(rating) AS number_ratings
FROM renting
WHERE date_renting >= '2019-01-01';

---

## 📌 Chapter 2 — Decision Making with Simple SQL Queries

---

### First Account per Country
Find when the first customer account was created for each country.

In [ ]:
SELECT country, MIN(date_account_start) AS first_account
FROM customers
GROUP BY country
ORDER BY first_account;

### Average Movie Ratings
Calculate avg rating, number of ratings, and number of rentals per movie.

In [ ]:
SELECT movie_id, 
       AVG(rating) AS avg_rating,
       COUNT(rating) AS number_ratings,
       COUNT(*) AS number_renting
FROM renting
GROUP BY movie_id
ORDER BY avg_rating DESC;

### Average Rating per Customer
Customers with more than 7 rentals, ordered by average rating.

In [ ]:
SELECT customer_id,
      AVG(rating),
      COUNT(rating),
      COUNT(*)
FROM renting
GROUP BY customer_id
HAVING COUNT(*) > 7
ORDER BY AVG(rating);

### Join Renting and Customers
Augment rentals with customer info, filter by country.

In [ ]:
-- All rentals with customer info
SELECT * FROM renting AS r
LEFT JOIN customers AS c ON r.customer_id = c.customer_id;

-- Only Belgium customers
SELECT * FROM renting AS r
LEFT JOIN customers AS c ON r.customer_id = c.customer_id
WHERE c.country = 'Belgium';

-- Average rating from Belgium
SELECT AVG(rating) FROM renting AS r
LEFT JOIN customers AS c ON r.customer_id = c.customer_id
WHERE c.country = 'Belgium';

### Revenue KPIs for 2018
Calculate total revenue, number of rentals, and active customers in 2018.

In [ ]:
SELECT 
	SUM(m.renting_price) AS revenue, 
	COUNT(*) AS number_rentals, 
	COUNT(DISTINCT r.customer_id) AS active_customers
FROM renting AS r
LEFT JOIN movies AS m ON r.movie_id = m.movie_id
WHERE date_renting BETWEEN '2018-01-01' AND '2018-12-31';

### Movies and Actors
List all actor-movie combinations.

In [ ]:
SELECT m.title, a.name
FROM actsin AS ai
LEFT JOIN movies AS m ON m.movie_id = ai.movie_id
LEFT JOIN actors AS a ON a.actor_id = ai.actor_id;

### Income per Movie
Calculate total revenue per movie, ordered by highest income.

In [ ]:
SELECT title, 
       SUM(renting_price) AS income_movie
FROM
       (SELECT m.title, m.renting_price
       FROM renting AS r
       LEFT JOIN movies AS m ON r.movie_id = m.movie_id) AS rm
GROUP BY title
ORDER BY income_movie DESC;

### Age of US Actors
Report oldest and youngest US actor/actress by gender.

In [ ]:
SELECT gender, MAX(year_of_birth), MIN(year_of_birth)
FROM (SELECT * FROM actors WHERE nationality = 'USA') AS a
GROUP BY gender;

### Favorite Movies for Customers Born in the 70s
Find top-rated movies among customers born between 1970–1979.

In [ ]:
SELECT m.title, count(*), AVG(r.rating)
FROM renting AS r
LEFT JOIN customers AS c ON c.customer_id = r.customer_id
LEFT JOIN movies AS m ON m.movie_id = r.movie_id
WHERE c.date_of_birth BETWEEN '1970-01-01' AND '1979-12-31'
GROUP BY m.title
HAVING COUNT(*) > 1
ORDER BY AVG(rating) DESC;

### Favorite Actors for Spanish Customers
Report actor popularity (views + avg rating) for customers from Spain.

In [ ]:
SELECT a.name, c.gender,
       COUNT(*) AS number_views, 
       AVG(r.rating) AS avg_rating
FROM renting AS r
LEFT JOIN customers AS c ON r.customer_id = c.customer_id
LEFT JOIN actsin AS ai ON r.movie_id = ai.movie_id
LEFT JOIN actors AS a ON ai.actor_id = a.actor_id
WHERE c.country = 'Spain'
GROUP BY a.name, c.gender
HAVING AVG(r.rating) IS NOT NULL AND COUNT(*) > 5
ORDER BY avg_rating DESC, number_views DESC;

### KPIs per Country (2019+)
Revenue, rentals, and average rating per country since 2019.

In [ ]:
SELECT 
	c.country,
	count(*) AS number_renting,
	AVG(r.rating) AS average_rating,
	SUM(m.renting_price) AS revenue
FROM renting AS r
LEFT JOIN customers AS c ON c.customer_id = r.customer_id
LEFT JOIN movies AS m ON m.movie_id = r.movie_id
WHERE r.date_renting >= '2019-01-01'
GROUP BY c.country;

---

## 📌 Chapter 3 — Advanced SQL: Subqueries & EXISTS

---

### Often Rented Movies (Nested Subquery)
List movies with more than 5 views using a subquery in WHERE.

In [ ]:
SELECT * FROM movies
WHERE movie_id IN
	(SELECT movie_id FROM renting
	GROUP BY movie_id
	HAVING COUNT(*) > 5);

### Frequent Customers
List customers who rented more than 10 movies.

In [ ]:
SELECT * FROM customers
WHERE customer_id IN
	(SELECT customer_id FROM renting
	GROUP BY customer_id
	HAVING COUNT(*) > 10);

### Movies with Rating Above Average
Find movies with average rating higher than the overall average.

In [ ]:
-- Overall average
SELECT AVG(rating) FROM renting;
-- R: 7.94

-- Movie titles above average
SELECT title FROM movies
WHERE movie_id IN
	(SELECT movie_id FROM renting
     GROUP BY movie_id
     HAVING AVG(rating) > 
		(SELECT AVG(rating) FROM renting));

### Correlated Subquery — Customer Behavior
Find customers who rented fewer than 5 movies (target for advertising).

In [ ]:
SELECT * FROM customers AS c
WHERE 5 >
	(SELECT count(*) FROM renting AS r
	WHERE r.customer_id = c.customer_id);

### Customers Who Gave Low Ratings
Find customers with minimum rating below 4.

In [ ]:
SELECT * FROM customers AS c
WHERE 4 >
	(SELECT MIN(rating) FROM renting AS r
	WHERE r.customer_id = c.customer_id);

### Movies with Correlated Subqueries
Find movies with more than 5 ratings AND movies with avg rating > 8.

In [ ]:
-- Movies with more than 5 ratings
SELECT * FROM movies AS m
WHERE 5 <
	(SELECT COUNT(rating) FROM renting AS r
	WHERE r.movie_id = m.movie_id);

-- Movies with avg rating > 8
SELECT * FROM movies AS m
WHERE 8 <
	(SELECT AVG(rating) FROM renting AS r
	WHERE r.movie_id = m.movie_id);

### EXISTS — Customers with At Least One Rating
Select all customers who gave at least one rating.

In [ ]:
SELECT * FROM customers AS c
WHERE EXISTS
	(SELECT * FROM renting AS r
	WHERE rating IS NOT NULL 
	AND r.customer_id = c.customer_id);

### EXISTS — Actors in Comedies
List actors who appear in at least one comedy, with count by nationality.

In [ ]:
-- List of actors in comedies
SELECT * FROM actors AS a
WHERE EXISTS
	(SELECT * FROM actsin AS ai
	 LEFT JOIN movies AS m ON m.movie_id = ai.movie_id
	 WHERE m.genre = 'Comedy'
	 AND ai.actor_id = a.actor_id);

-- Count by nationality
SELECT a.nationality, count(*)
FROM actors AS a
WHERE EXISTS
	(SELECT ai.actor_id FROM actsin AS ai
	 LEFT JOIN movies AS m ON m.movie_id = ai.movie_id
	 WHERE m.genre = 'Comedy'
	 AND ai.actor_id = a.actor_id)
GROUP BY a.nationality;

### UNION / INTERSECT — Young Non-American Actors
Combine sets to find actors not from USA, born after 1990.

In [ ]:
-- UNION: not from USA OR born after 1990
SELECT name, nationality, year_of_birth FROM actors WHERE nationality <> 'USA'
UNION
SELECT name, nationality, year_of_birth FROM actors WHERE year_of_birth > 1990;

-- INTERSECT: not from USA AND born after 1990
SELECT name, nationality, year_of_birth FROM actors WHERE nationality <> 'USA'
INTERSECT
SELECT name, nationality, year_of_birth FROM actors WHERE year_of_birth > 1990;

### INTERSECT — Dramas with High Ratings
Find dramas with average rating higher than 9.

In [ ]:
SELECT * FROM movies
WHERE movie_id IN
   (SELECT movie_id FROM movies WHERE genre = 'Drama'
    INTERSECT
    SELECT movie_id FROM renting
    GROUP BY movie_id HAVING AVG(rating) > 9);

---

## 📌 Chapter 4 — OLAP Queries: CUBE, ROLLUP & GROUPING SETS

---

### OLAP Overview

**OLAP (On-line Analytical Processing)** operators extend GROUP BY for multi-dimensional analysis:

| Operator | Description |
|----------|-------------|
| **CUBE** | All possible combinations of grouping columns + grand total |
| **ROLLUP** | Hierarchical subtotals (left to right) + grand total |
| **GROUPING SETS** | Custom-defined grouping combinations (most flexible) |


### CUBE — Customer Count by Country and Gender
Extract a pivot table of customer counts by all combinations of gender and country.

In [ ]:
SELECT gender, country, count(*)
FROM customers
GROUP BY CUBE(gender, country)
ORDER BY country;

### CUBE — Movie Categories
Count movies for all combinations of genre and year of release.

In [ ]:
SELECT genre, year_of_release, count(*)
FROM movies
GROUP BY CUBE(genre, year_of_release)
ORDER BY year_of_release;

### CUBE — Average Ratings by Country and Genre
Calculate average ratings for all aggregation levels of country and genre.

In [ ]:
SELECT country, genre, AVG(r.rating) AS avg_rating
FROM renting AS r
LEFT JOIN movies AS m ON m.movie_id = r.movie_id
LEFT JOIN customers AS c ON r.customer_id = c.customer_id
GROUP BY CUBE(country, genre);
-- Overall average: 7.94

### ROLLUP — Customer Count Hierarchy
Total customers → per country → per country+gender.

In [ ]:
SELECT country, gender, COUNT(*)
FROM customers
GROUP BY ROLLUP(country, gender)
ORDER BY (country, gender);

### ROLLUP — Genre Preferences by Country
Average rating and rental count per country and genre, with ROLLUP subtotals.

In [ ]:
SELECT c.country, m.genre, AVG(r.rating) AS avg_rating, COUNT(*) AS num_rating
FROM renting AS r
LEFT JOIN movies AS m ON m.movie_id = r.movie_id
LEFT JOIN customers AS c ON r.customer_id = c.customer_id
GROUP BY ROLLUP(c.country, m.genre)
ORDER BY c.country, m.genre;

### GROUPING SETS — Most Flexible OLAP Operator

```sql
GROUP BY GROUPING SETS ((country, genre), (country), (genre), ())
```

- Equivalent to `GROUP BY CUBE(country, genre)`
- But you choose **exactly** which combinations to include
- Replaces a UNION over multiple GROUP BY queries


### GROUPING SETS — Actor Nationality and Gender
Count actors by nationality, by gender, and grand total — in one query.

In [ ]:
SELECT nationality, gender, COUNT(*)
FROM actors
GROUP BY GROUPING SETS((nationality), (gender), ());

### GROUPING SETS — Rating by Country and Gender
Report average rating for all pivot table combinations of country and gender.

In [ ]:
SELECT c.country, c.gender, AVG(r.rating)
FROM renting AS r
LEFT JOIN customers AS c ON r.customer_id = c.customer_id
GROUP BY GROUPING SETS((country, gender), (country), (gender), ());

### Customer Preference for Genres
Find best-rated genres for movies with at least 3 ratings since 2018.

In [ ]:
SELECT genre,
	   AVG(rating) AS avg_rating,
	   COUNT(rating) AS n_rating,
       COUNT(*) AS n_rentals,
	   COUNT(DISTINCT m.movie_id) AS n_movies 
FROM renting AS r
LEFT JOIN movies AS m ON m.movie_id = r.movie_id
WHERE r.movie_id IN (
	SELECT movie_id FROM renting
	GROUP BY movie_id HAVING COUNT(rating) >= 3)
AND r.date_renting >= '2018-01-01'
GROUP BY genre
ORDER BY AVG(rating) DESC;

### Customer Preference for Actors — CUBE Pivot
Analyze ratings by actor nationality and gender across all aggregation levels.

In [ ]:
SELECT a.nationality, a.gender,
	   AVG(r.rating) AS avg_rating,
	   COUNT(r.rating) AS n_rating,
	   COUNT(*) AS n_rentals,
	   COUNT(DISTINCT a.actor_id) AS n_actors
FROM renting AS r
LEFT JOIN actsin AS ai ON ai.movie_id = r.movie_id
LEFT JOIN actors AS a ON ai.actor_id = a.actor_id
WHERE r.movie_id IN (
	SELECT movie_id FROM renting
	GROUP BY movie_id HAVING COUNT(rating) >= 4)
AND r.date_renting >= '2018-04-01'
GROUP BY CUBE(a.nationality, a.gender);